# Solution: 02_claude_code_harness

이 노트북은 **Claude Code**의 핵심 아키텍처 패턴인 **5계층 프롬프트 조립**, **5대 컨텍스트 압축 파이프라인**, **Amnesia Guard(기억상실 방지 복구)**, 및 **Stop Hooks 자가 수정 게이트**를 LangChain 및 LangGraph 환경에서 직접 구현하고 검증하는 실습 교안입니다.


## 🛠️ Step 0. 환경 세팅 (Environment Setup)

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`), 그리고 통합 LLM 팩토리를 초기화합니다.

In [ ]:
import os
import sys
import re
import json
import time
import asyncio
import warnings
import nest_asyncio
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# 1. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 2. 프로젝트 루트 상향 동적 탐색 (어느 서브 폴더에 노트북이 있어도 100% 작동)
def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), ".."))

project_root = find_project_root()
os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. 환경변수 명시적 로드
dotenv_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path, override=True)

# LangSmith / LangChain Tracing 경고 제어
os.environ["LANGCHAIN_TRACING_V2"] = "false"

print(f"✅ Working Directory: {os.getcwd()}")
print(f"✅ Project Root: {project_root}")

# 4. 통합 Chat Model Factory 및 핵심 모듈 임포트
from app.utils import init_chat_model, normalize_content
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from modules.common.agent_tracer import AgentTracer

# 5. 이전 코드 호환용 get_llm 헬퍼 (노트북 내 잔여 get_llm 호출 완충)
def get_llm(model_name="gemini-3.7-flash", **kwargs):
    return init_chat_model(model=model_name, **kwargs)

# 6. 기본 LLM 및 트레이서 초기화
llm = init_chat_model(model="gemini-3.7-flash", temperature=0.0)
tracer = AgentTracer(log_dir="./artifacts/logs", verbose=True)

print(f"✅ LLM Model Initialized: {getattr(llm, 'model', getattr(llm, 'model_name', 'gemini-3.7-flash'))}")
print("✅ Setup completed successfully!")

## 핵심 실습 1: 5계층 프롬프트 조립 (5-Layer Prompt Assembly)

In [ ]:
# ===== 셀 1: 16대 모듈 프롬프트 조립기 초기화 =====
import os
from app.middleware.prompt.prompt_assembler import (
    PromptAssembler,
    prepend_user_context,
    attach_mcp_delta_if_needed,
    get_frozen_git_snapshot,
)

# 1. 기본 시스템 규칙 및 세션 컨텍스트 초기화
PROMPT_MD_PATH = "app/prompts/SUPERVISOR.py"
system_rules_content = "You are a professional software engineer and pair programming assistant."
try:
    from app.prompts import SUPERVISOR_SYSTEM_PROMPT
    system_rules_content = SUPERVISOR_SYSTEM_PROMPT
except Exception:
    pass

session_ctx = {
    "cwd": os.getcwd(),
    "session_id": "session_claude_code_001",
    "os": os.name,
    "user_permission": "FULL_ACCESS",
    "active_project": "frontier_agent_analysis",
    "git_status": "On branch main, clean working tree",
}

# 2. 16대 표준 모듈 기반 5계층 PromptAssembler 생성
assembler_cc = PromptAssembler(
    system_rules=system_rules_content,
    tool_schemas=[
        {"name": "file_writer", "description": "로컬 파일 생성/수정", "args": {"path": "str", "content": "str"}},
        {"name": "bash_command", "description": "WSL 쉘 명령어 실행", "args": {"cmd": "str"}},
        {"name": "file_read", "description": "소스 코드 파일 읽기", "args": {"path": "str"}},
    ],
    claude_code_modules=True,      # ✨ 16대 표준 모듈 활성화
    language="Korean",
    output_style="Concise",
    mcp_delta_enabled=True,
)

messages_cc = assembler_cc.assemble(
    user_input="위키피디아 MCP 서버를 활용해서 검색 툴을 어떻게 호출하나요?",
    session_context=session_ctx,
)

print(f"=== 메시지 수: {len(messages_cc)} ===\n")
print("[Static (L1+L2)] 길이:", len(messages_cc[0].content), "chars")
print("[Dynamic (L3+L4+L5)] 길이:", len(messages_cc[1].content), "chars")


In [ ]:
# ===== 셀 2: Static 영역 검증 (L1 Global Constitution + L2 Tool Guide) =====
print("=" * 70)
print("📌 [Message 1: Static — L1 Global Constitution + L2 Tool Guide]")
print("=" * 70)
print(messages_cc[0].content[:3000])  # 앞부분만 출력
print("\n... (이하 생략)")
print(messages_cc[1].content[:3000])  # 앞부분만 출력
print("\n... (이하 생략)")
print(messages_cc[2].content[:3000])  # 앞부분만 출력

In [ ]:
# ===== 셀 3: User Context Bypass Injection 테스트 =====
from langchain_core.messages import HumanMessage

test_messages = [
    SystemMessage(content="Static system prompt..."),
    SystemMessage(content="Dynamic prompt..."),
    HumanMessage(content="프로젝트의 테스트 커버리지를 확인해주세요."),
]

# CLAUDE.md + Git 스냅샷 우회 주입
bypassed = prepend_user_context(
    test_messages,
    claude_md_content="항상 pytest를 사용하세요.\n코드 리뷰 없이 main 브랜치에 직접 푸시하지 마세요.",
    git_snapshot="M  app/main.py\n\nRecent commits:\nabc1234 fix: DB config env vars",
)

print("=" * 70)
print("📌 [User Context Bypass: messages[0]에 <system-reminder> 주입 결과]")
print("=" * 70)
for i, m in enumerate(bypassed):
    print(f"\n[Message {i+1}: {m.__class__.__name__}]")
    print("-" * 40)
    print(m.content)

## 핵심 실습 2: 5대 컨텍스트 압축 파이프라인 (Compactor Pipeline & Amnesia Guard)

### 📊 5대 컨텍스트 압축 파이프라인 카드
<div style="background-color: #0F172A; border: 1px solid #334155; border-radius: 12px; padding: 20px; font-family: 'Malgun Gothic', 'Pretendard', sans-serif; color: #E2E8F0; margin: 15px 0;">
<h3 style="color: #F8FAFC; margin-top: 0; border-bottom: 2px solid #1E293B; padding-bottom: 10px; font-size: 1.15rem;">⚡ Claude Code 5대 컨텍스트 압축 파이프라인 (Compactor Pipeline)</h3>
<h4 style="color: #38BDF8; margin: 15px 0 10px 0; font-size: 1rem;">Phase 1: 매 턴 선제적 압축 (Cost $0 최적화)</h4>
<table style="width: 100%; border-collapse: separate; border-spacing: 10px; background: transparent; margin: 0; padding: 0; border: none;">
<tr>
<td style="width: 50%; background-color: #1E293B; border-left: 4px solid #38BDF8; padding: 12px; border-radius: 6px; vertical-align: top;">
<div style="font-weight: bold; color: #38BDF8; font-size: 0.95rem;">✂️ 1. Snip Compact</div>
<div style="font-size: 0.85rem; color: #94A3B8; margin-top: 4px;">• 2턴 경과된 과거 툴 결과를 1줄 스텁으로 치환</div>
<div style="font-size: 0.8rem; color: #34D399; font-weight: bold; margin-top: 6px;">[토큰 절감 93%]</div>
</td>
<td style="width: 50%; background-color: #1E293B; border-left: 4px solid #F59E0B; padding: 12px; border-radius: 6px; vertical-align: top;">
<div style="font-weight: bold; color: #F59E0B; font-size: 0.95rem;">📦 2. Microcompact</div>
<div style="font-size: 0.85rem; color: #94A3B8; margin-top: 4px;">• 5,000자 초과 툴 로그를 디스크 스왑 파일로 덤프</div>
<div style="font-size: 0.8rem; color: #34D399; font-weight: bold; margin-top: 6px;">[프롬프트 절감 98%]</div>
</td>
</tr>
<tr>
<td style="width: 50%; background-color: #1E293B; border-left: 4px solid #A78BFA; padding: 12px; border-radius: 6px; vertical-align: top;">
<div style="font-weight: bold; color: #A78BFA; font-size: 0.95rem;">📂 3. Context Collapse</div>
<div style="font-size: 0.85rem; color: #94A3B8; margin-top: 4px;">• 3회 이상 연속 탐색 툴 실행을 1개 블록으로 접기</div>
<div style="font-size: 0.8rem; color: #34D399; font-weight: bold; margin-top: 6px;">[메시지 절감 55%]</div>
</td>
<td style="width: 50%; background-color: #1E293B; border-left: 4px solid #EC4899; padding: 12px; border-radius: 6px; vertical-align: top;">
<div style="font-weight: bold; color: #EC4899; font-size: 0.95rem;">🤖 4. Auto-Compact</div>
<div style="font-size: 0.85rem; color: #94A3B8; margin-top: 4px;">• 임계치 초과 시 4대 영역 요약 + Amnesia Guard 복구</div>
<div style="font-size: 0.8rem; color: #EC4899; font-weight: bold; margin-top: 6px;">[+ Amnesia Guard 주입]</div>
</td>
</tr>
</table>
<h4 style="color: #EF4444; margin: 20px 0 10px 0; font-size: 1rem;">Phase 2: 사후 방화벽 (Emergency Recovery Firewall)</h4>
<div style="background-color: #1E293B; border: 1.5px solid #EF4444; padding: 15px; border-radius: 8px;">
<div style="font-weight: bold; color: #EF4444; font-size: 1rem;">⚡ 5. Reactive Compact</div>
<div style="font-size: 0.85rem; color: #CBD5E1; margin-top: 6px; line-height: 1.5;">
API 413 prompt_too_long 오류 발생 시 <b>Silent Withholding</b>으로 오류 팝업을 삼키고,<br>
오래된 대화의 20%를 강제 절단 후 Amnesia Guard 복구 어태치먼트를 주입하여 즉시 백그라운드 재요청합니다.
</div>
</div>
</div>

### 📖 원본 핵심 발췌 명세
- **Snip Compact**: 매 턴 시작 전 오래된 과거 툴 결과(2턴 경과)를 1줄 스텁(`[Output snipped]`)으로 변환하여 토큰 절감.
- **Microcompact**: 5,000자 초과 대용량 툴 결과(`bash_cmd` 로그 등)를 디스크 임시 파일(`.claude/swaps/swap_*.txt`)로 덤프 저장 후 포인터 스텁 주입.
- **Context Collapse**: 수많은 탐색/리서치 툴 실행 블록(3회 이상 연속)을 1개의 접힌 메타데이터 블록(`[Context Collapsed: N research steps]`)으로 병합.
- **Auto-Compact**: 토큰 한도 초과 시 포크 에이전트가 4대 영역 구조화 요약문 생성 + **Amnesia Guard(최근 파일 5개 스냅샷 & Active Plan)**를 복구 어태치먼트로 주입.
- **Reactive Compact**: API 413 `prompt_too_long` 에러 발생 시 에러를 은폐(Silent Withholding)하고 오래된 메시지 20%를 자른 뒤 즉시 자동 재시도.


In [ ]:
from modules.claude_code.compactor import (
    SnipCompactor,
    MicroCompactor,
    ContextCollapse,
    AutoCompactor,
    ReactiveCompactor,
    create_compactor_middleware
)
from modules.claude_code.amnesia_guard import AmnesiaGuardMiddleware, create_amnesia_guard_middleware

amnesia_guard = AmnesiaGuardMiddleware(max_restore_files=5)
print("Compactor 파이프라인 5종 모듈 및 AmnesiaGuardMiddleware 초기화 성공!")

### ✂️ [시나리오 1] Snip Compact (오래된 툴 결과 가위질 절삭)

#### 📌 Snip Compact란 무엇인가요?
에이전트와 장시간 개발 대화를 나누다 보면, **3~4턴 전에 조회했던 수백 줄짜리 소스 코드(`file_read`)나 터미널 실행 로그**가 대화 기록(Message Array)에 계속 누적됩니다.

방금 읽은 최근 코드는 바로 다음 작업에 필요하지만, **몇 턴이나 지난 과거 툴 결과 원문**을 매 턴마다 LLM에게 다시 전송하는 것은 엄청난 **토큰 비용 낭비이자 속도 저하의 주원인**이 됩니다.

`Snip Compact`는 매 턴이 시작되기 직전, 경과 시간(Turn Age)이 오래된 과거 툴 실행 결과 텍스트를 **`[Tool result snipped: Executed 'file_read' successfully (1,200 chars)]`와 같이 1줄짜리 요약 스텁(Stub)으로 가위질(Snip)하여 토큰을 절감**해 주는 예방적 압축 기술입니다.

---

#### 💡 왜 사용해야 하나요? (핵심 특징)
1. 💰 **비용 $0 기반 토큰 절감**: 과거 수천 줄의 소스 코드가 1줄 스텁으로 줄어들어 입력 토큰을 **90% 이상 대폭 절감**합니다.
2. ⚡ **응답 속도 단축**: 입력 프롬프트 길이가 가벼워지므로 LLM의 첫 토큰 생성 시간(TTFT)이 획기적으로 빨라집니다.
3. 🛡️ **최근 컨텍스트 보존**: 가장 최근(지정한 threshold 이내) 실행된 툴 결과는 절삭하지 않고 **원본 텍스트 그대로 안전하게 유지**합니다.

---

#### 🔍 아래 실습 코드에서 관찰할 포인트
- **4턴 전 툴 출력 (`Message [2]`)**: `app/main.py` 파일(1,200자)을 읽었던 과거 기록 ➔ **`[Tool result snipped...]` 1줄 스텁으로 가위질됨!**
- **1턴 전 툴 출력 (`Message [6]`)**: `app/utils.py` 파일을 방금 읽은 최근 기록 ➔ **소스 코드 원본 그대로 보존됨!**


In [ ]:
snip_compactor = SnipCompactor(age_threshold=2)

messages_s1 = [
    SystemMessage(content="시스템 지침: 개발 보조 에이전트"),
    
    # [Turn 1: 과거 턴 - 3턴 전] 사용자 요청 ➔ AI 툴 호출 ➔ 1,200자 대용량 읽기 결과 ➔ AI 분석 응답
    HumanMessage(content="app/main.py 설정을 읽어서 DB 연동 방식을 확인해줘."),
    AIMessage(
        content="app/main.py 파일 내용을 확인하겠습니다.",
        tool_calls=[{"name": "file_read", "args": {"path": "app/main.py"}, "id": "c1"}]
    ),
    ToolMessage(
        content="# app/main.py\n" + "import os\nimport sys\ndef init_db(): pass\n" * 30, # 과거 툴 결과 (1,200자 -> Snip 대상)
        tool_call_id="c1", 
        name="file_read"
    ),
    AIMessage(content="app/main.py를 확인한 결과, DB_HOST가 하드코딩되어 있습니다."),
    
    # [Turn 2: 과거 턴 - 2턴 전] 코드 수정 요청 ➔ AI 수정 완료 응답
    HumanMessage(content="DB 설정 파라미터를 env 환경변수로 변경해줘."),
    AIMessage(content="app/main.py 15번 라인의 DB_HOST를 os.getenv('DB_HOST')로 변경했습니다."),
    
    # [Turn 3: 최근 턴 - 1턴 전] 최근 유틸 읽기 ➔ AI 툴 호출 ➔ 툴 결과 ➔ AI 분석 응답
    HumanMessage(content="유틸리티 모듈 app/utils.py 도 읽어줘."),
    AIMessage(
        content="app/utils.py 파일 내용을 읽어오겠습니다.",
        tool_calls=[{"name": "file_read", "args": {"path": "app/utils.py"}, "id": "c2"}]
    ),
    ToolMessage(
        content="# app/utils.py\ndef parse_json(): return {}", # 최근 툴 결과 (age_threshold 이내 -> 원본 보존)
        tool_call_id="c2", 
        name="file_read"
    ),
    AIMessage(content="app/utils.py의 JSON 파싱 헬퍼 함수 구현을 확인했습니다."),
    
    # [Turn 4: 현재 턴 - 0턴 전] 현재 사용자 질문
    HumanMessage(content="현재 변경 상태는 어떠한가요?")
]

print("==========================================================")
print("📌 [Snip Compact 적용 전 대화 스트림 pretty_print]")
print("==========================================================")
for m in messages_s1:
    m.pretty_print()

In [ ]:
compacted_s1, modified_s1 = snip_compactor.compact(messages_s1)

print("\n" + "==========================================================")
print(f"✂️ [Snip Compact 적용 후 대화 스트림 pretty_print (Modified: {modified_s1})]")
print("==========================================================")
for m in compacted_s1:
    m.pretty_print()

### 📦 [시나리오 2] Microcompact (대용량 터미널/빌드 로그 디스크 스왑)

#### 📌 Microcompact란 무엇인가요?
에이전트가 `pytest` 단위 테스트나 `npm build` 스크립트를 실행하면 **수천 줄(5,000자 이상)에 달하는 거대한 터미널 라이브 출력 로그**가 반환될 때가 있습니다.

이 거대한 텍스트를 대화 기록(Prompt)에 그대로 집어넣으면 단 한 번의 툴 호출만으로도 **토큰 한도가 즉시 폭발**하게 됩니다.

`Microcompact`는 5,000자가 넘어가는 거대한 툴 출력 결과가 들어오면, 원문 전체를 프롬프트에 넣지 않고 **로컬 디스크 파일(`.claude/swaps/swap_*.txt`)로 덤프 저장**한 뒤, 프롬프트에는 **`[대용량 터미널 출력 디스크 스왑 완료: .claude/swaps/swap_*.txt (6,500자)]` 포인터 스텁만 잔류**시키는 기술입니다.

---

#### 💡 왜 사용해야 하나요? (핵심 특징)
1. 💣 **토큰 폭발(Token Explosion) 차단**: 단 한 번의 대용량 로그 출력으로 대화창이 마비되는 현상을 방지합니다.
2. 💾 **100% 영구 보존**: 프롬프트 토큰은 아끼면서도, 실제 6,500자 원문 로그는 로컬 디스크 파일에 **100% 안전하게 저장**됩니다.
3. 🔍 **필요시 재참조 가능**: 에이전트나 개발자가 특정 에러 라인을 자세히 봐야 할 경우 `file_read`로 해당 스왑 파일만 언제든지 다시 읽을 수 있습니다.

---

#### 🔍 아래 실습 코드에서 관찰할 포인트
- **pytest 테스트 실행 로그**: 6,500자에 달하는 방대한 터미널 로그 생성
- **스왑 처리 후 출력**: 프롬프트에는 스왑 포인터 스텁만 들어가고, `.claude/swaps/` 폴더에 실제 `.txt` 로그 파일이 덤프 저장된 것을 확인!


In [ ]:
swap_dir = "./.claude/swaps"
micro_compactor = MicroCompactor(max_chars=3000, swap_dir=swap_dir)

# 약 8,500자 크기의 대용량 터미널 pytest 실행 로그 생성
large_test_log = "=== pytest v8.2.0 test run start ===\n" + ("test_auth.py::test_login PASSED [ 1%]\ntest_auth.py::test_token PASSED [ 2%]\n" * 120)

messages_s2 = [
    SystemMessage(content="시스템 지침: 테스트 실행 에이전트"),
    
    # [Turn 1] 사용자 요청 ➔ AI 툴 호출 제안 ➔ 8,500자 대용량 로그 출력 ➔ AI 결과 보고
    HumanMessage(content="전체 단위 테스트 pytest 스크립트를 실행해줘."),
    AIMessage(
        content="터미널에서 pytest 명령어를 실행하여 단위 테스트를 구동하겠습니다.",
        tool_calls=[{"name": "bash_command", "args": {"cmd": "pytest"}, "id": "c_pytest"}]
    ),
    ToolMessage(
        content=large_test_log, # 8,500자 대용량 로그 (3,000자 초과 -> 디스크 스왑 대상)
        tool_call_id="c_pytest", 
        name="bash_command"
    ),
    AIMessage(content="전체 단위 테스트가 성공적으로 통과했습니다 (240 tests passed).")
]

print("==========================================================")
print("📌 [Microcompact 적용 전 대화 스트림 pretty_print]")
print("==========================================================")
for m in messages_s2:
    m.pretty_print()

In [ ]:
compacted_s2, modified_s2 = micro_compactor.compact(messages_s2)

print("\n" + "==========================================================")
print(f"📦 [Microcompact 적용 후 대화 스트림 pretty_print (Modified: {modified_s2})]")
print("==========================================================")
for m in compacted_s2:
    m.pretty_print()

# 디스크 스왑 폴더 실제 생성 파일 검증
import os
if os.path.exists(swap_dir):
    swap_files = os.listdir(swap_dir)
    print(f"\n📂 [디스크 스왑 폴더 '{swap_dir}' 생성 파일 목록]: {swap_files}")

In [ ]:
print(f"\n  - [디스크 스왑 폴더 '{swap_dir}' 생성 파일]: {os.listdir(swap_dir)}")

### 📂 [시나리오 3] Context Collapse (코드 탐색 툴 호출 블록 접기)

#### 📌 Context Collapse란 무엇인가요?
개발자가 "인증 모듈 관련 코드를 찾아서 리팩토링해줘"라고 요청하면, 에이전트는 코드를 직접 수정하기 전에 **`grep_search` ➔ `glob_search` ➔ `file_read` ➔ `file_read`**와 같이 **여러 번 연속으로 탐색 및 조회 툴을 실행**합니다.

수많은 탐색 툴 호출 메시지들이 대화창을 가득 채우면 **진짜 중요한 사용자의 지시사항과 최종 작성할 소스 코드의 맥락이 묻히게 됩니다.**

`Context Collapse`는 3회 이상 연속으로 실행된 중간 코드 탐색 과정(Research Steps)을 **단 1개의 접힌 스냅샷 메시지(`[Context Collapsed: 6 research steps...]`)로 통합 접기**하여 대화 이력을 깔끔하게 정돈하는 기술입니다.

---

#### 💡 원본 맥락은 복원할 수 있나요? (핵심 포인트)
- **100% 원문 복원 기능 지원**: 메시지를 접을 때 6단계의 세부 탐색 원본을 **`.claude/swaps/collapse_snap_*.txt` 디스크 스냅샷**으로 자동 저장합니다.
- 에이전트가 탐색 과정에서의 세부 코드를 다시 확인해야 할 경우, 해당 스냅샷 파일을 읽어 **원본 맥락을 언제든지 100% 원복**할 수 있습니다.

---

#### 🔍 아래 실습 코드에서 관찰할 포인트
- **접히기 전**: 탐색 및 조회 과정 툴 메시지 9개가 길게 늘어서 있음
- **접힌 후**: 중간 6개 탐색 메시지가 `[Context Collapsed: 6 research steps | Raw Snapshot: .claude/swaps/...]` 단 1개 메시지로 깔끔하게 축소됨!


In [ ]:
context_collapse = ContextCollapse(min_consecutive=3)

messages_s3 = [
    SystemMessage(content="시스템 지침: 코드 탐색 에이전트"),
    HumanMessage(content="전체 프로젝트에서 AuthMiddleware 클래스 위치를 찾고 소스를 읽어줘."),
    AIMessage(content="grep 툴로 AuthMiddleware 검색 중...", tool_calls=[{"name": "grep_search", "args": {}, "id": "t1"}]),
    ToolMessage(content="Found in app/middleware/auth.py", tool_call_id="t1", name="grep_search"),
    AIMessage(content="glob 툴로 관련 라우터 탐색 중...", tool_calls=[{"name": "glob_search", "args": {}, "id": "t2"}]),
    ToolMessage(content="Found app/api/auth_router.py", tool_call_id="t2", name="glob_search"),
    AIMessage(content="auth.py 파일 내용 읽는 중...", tool_calls=[{"name": "file_read", "args": {}, "id": "t3"}]),
    ToolMessage(content="class AuthMiddleware: pass", tool_call_id="t3", name="file_read"),
    HumanMessage(content="검색된 파일들을 기반으로 인증 토큰 검증 로직을 구현해줘.")
]

print("==========================================================")
print(f"📌 [Context Collapse 적용 전 대화 스트림 (전체 {len(messages_s3)}개)]")
print("==========================================================")
for m in messages_s3:
    m.pretty_print()

In [ ]:
compacted_s3, modified_s3 = context_collapse.compact(messages_s3)

print("\n" + "==========================================================")
print(f"📂 [Context Collapse 적용 후 대화 스트림 (축소 후 {len(compacted_s3)}개, Modified: {modified_s3})]")
print("==========================================================")
for m in compacted_s3:
    m.pretty_print()

### 🤖 [시나리오 4] Auto-Compact & Amnesia Guard (선제적 딥 요약 + 기억상실 방지 복구)

#### 📌 Auto-Compact & Amnesia Guard란 무엇인가요?
대화가 매우 길어져 전체 누적 토큰이 모델의 안전 임계치(Context Window Threshold)에 도달하면, 과거 대화 이력 전체를 **LLM 4대 영역(Primary Goal, Key Decisions, Workspace Delta, Next Action)으로 구조화 딥 요약(Auto-Compact)**하여 대화창을 리셋합니다.

하지만 대화가 요약문 1개로 축소되면 에이전트는 **방금 전까지 읽고 수정하던 소스 코드나 현재 진행 중인 플랜(Plan)을 잊어버리는 '기억상실(Amnesia)'** 상태에 빠지게 됩니다.

`Amnesia Guard`는 요약이 끝난 직후 에이전트가 방금 다루던 **최근 핵심 파일 스냅샷(최대 5개)과 진행 중인 Active Plan**을 **`[Compaction Amnesia Guard: Restoring Recent Work Context]` 어태치먼트 SystemMessage로 즉시 복구 주입**하여 에이전트의 개발 연속성을 완벽하게 보장합니다.

---

#### 💡 왜 함께 사용해야 하나요? (핵심 효과)
1. 🧠 **기억상실(Amnesia) 완벽 방지**: 대화가 축소되어도 에이전트가 방금 읽던 코드 원본과 계획을 잊지 않습니다.
2. 🔄 **툴 재호출 낭비 $0**: 기억상실로 인해 "아까 읽은 파일을 다시 읽겠습니다" 하고 `file_read` 툴을 재호출하는 토큰 및 시간 낭비를 완전히 차단합니다.
3. 📋 **구조화된 맥락 전수**: 4대 영역 구조화 요약문 덕분에 장시간 대화에도 에이전트가 초심(Goal)을 잃지 않습니다.

---

#### 🔍 아래 실습 코드에서 관찰할 포인트
- **Auto-Compact 완료 후**: 이전 7개 대화가 `Previous Conversation Summary:` 요약문 1개로 축소됨
- **Amnesia Guard 주입 확인**: 요약문 바로 밑에 `PROMPT.md`, `MCP.md` 소스 코드 전체와 `Active Plan`이 복구 어태치먼트로 자동 삽입된 것을 확인!


In [ ]:
# 1. Amnesia Guard 파일 및 플랜 감시 세팅
amnesia_guard.track_file_access("app/prompts/PROMPT.md")
amnesia_guard.track_file_access("app/prompts/MCP.md")
amnesia_guard.set_active_plan("1단계: Amnesia Guard 복구 어태치먼트 주입 검증\n2단계: AutoCompactor 요약 성능 평가")
# 2. AutoCompactor 초기화 (예제 대화량인 59토큰보다 작은 threshold_tokens=30 으로 설정)

auto_compactor = AutoCompactor(llm=llm, threshold_tokens=30, amnesia_guard=amnesia_guard)

# 3. 대화 이력 구성 (약 59 토큰)
messages_s4 = [
    SystemMessage(content="Layer 1: 정적 시스템 지침 (PROMPT.md)"),
    SystemMessage(content="Layer 4: 동적 세션 정보 (CWD, MCP.md)"),
    HumanMessage(content="위키피디아 MCP 서버 설정 문서를 검토하려고 합니다."),
    AIMessage(content="app/prompts/MCP.md 파일을 읽었습니다. 위키피디아 npx 연결 명령어가 포함되어 있습니다."),
    HumanMessage(content="좋습니다. 해당 서버의 연결 실행 명령어가 무엇인가요?"),
    AIMessage(content="연결 명령어는 'npx -y wikipedia-mcp' 입니다. 이제 검색 테스트를 수행할 수 있습니다."),
    HumanMessage(content="검색 테스트를 진행해 주세요.")
]

In [ ]:
print("==========================================================")
print(f"📌 [Auto-Compact 적용 전 대화 스트림 (전체 {len(messages_s4)}개)]")
print("==========================================================")
for m in messages_s4:
    m.pretty_print()

In [ ]:
# 4. 토큰 초과 자동 요약 + Amnesia Guard 복구 어태치먼트 주입 실행 (force=True 옵션도 가능)
compacted_s4, modified_s4 = auto_compactor.compact_if_needed(messages_s4, force=True)
print("\n" + "==========================================================")
print(f"🤖 [Auto-Compact + Amnesia Guard 적용 후 대화 스트림 (Modified: {modified_s4})]")
print("==========================================================")
for m in compacted_s4:
    m.pretty_print()

### ⚡ [시나리오 5] Reactive Compact (API 413 오버플로우 방화벽 & 자동 재시도)

#### 📌 Reactive Compact란 무엇인가요?
선제적 압축(Phase 1)에도 불구하고 사용자가 예기치 못하게 거대한 데이터/파일을 입력하거나 로그가 유입되면, **LLM API 서버로부터 HTTP 413 `prompt_too_long` (컨텍스트 오버플로우) 오류**를 수신하게 됩니다.

일반적인 서비스라면 화면에 붉은색 에러 팝업창을 띄우고 사용자의 대화가 끊기겠지만, Claude Code는 **Reactive Compact 사후 방화벽**이 작동합니다.

`Reactive Compact`는 API 413 오류가 발생하는 순간 **Silent Withholding 메커니즘으로 에러를 에러창 없이 백그라운드에서 삼킨 뒤**, 오래된 대화 이력의 20%를 강제 절단(Slice)하고 Amnesia Guard 복구 어태치먼트를 결합하여 **사용자 몰래 즉시 백그라운드 재요청(Auto-Retry)**을 수행하는 기술입니다.

---

#### 💡 왜 사용해야 하나요? (핵심 효과)
1. 🛡️ **끊김 없는 사용자 경험 (Seamless UX)**: 에러 팝업창으로 사용자의 작업 흐름이 방해받는 것을 완벽하게 방지합니다.
2. 🔄 **자율적 세션 복구**: 413 오류 시 에이전트 스스로 대화의 꼬리를 쳐내고 자율 복원하여 대화를 이어갑니다.

---

#### 🔍 아래 실습 코드에서 관찰할 포인트
- **413 오류 유발 시**: 오류 팝업이 출력되는 대신 `Silent Withholding` 요약 메시지가 주입됨
- **자동 복구 후**: 대화 꼬리 20%가 슬라이싱되고 세션이 안전하게 정상 복원된 상태를 확인!


In [ ]:
reactive_compactor = ReactiveCompactor(slice_ratio=0.20, amnesia_guard=amnesia_guard)

messages_s5 = [
    SystemMessage(content="Layer 1: 시스템 지침"),
    HumanMessage(content="1턴 질의: 프로젝트 구조 확인"),
    AIMessage(content="1턴 응답: 폴더 구조 분석 완료"),
    HumanMessage(content="2턴 질의: 모듈 작성"),
    AIMessage(content="2턴 응답: 모듈 작성 완료"),
    HumanMessage(content="3턴 질의: 데이터 분석"),
    AIMessage(content="3턴 응답: 데이터 분석 완료"),
    HumanMessage(content="4턴 질의: [대용량 페이로드 전송으로 API 413 오버플로우 에러 유발]")
]

print("==========================================================")
print(f"📌 [API 413 오버플로우 감지 - 적용 전 대화 스트림 (전체 {len(messages_s5)}개)]")
print("==========================================================")
for m in messages_s5:
    m.pretty_print()

In [ ]:
recovered_s5 = reactive_compactor.handle_overflow(messages_s5)

print("\n" + "==========================================================")
print(f"⚡ [Reactive Compact Silent 복구 후 대화 스트림 (절단 후 {len(recovered_s5)}개)]")
print("==========================================================")
for m in recovered_s5:
    m.pretty_print()

## 핵심 실습 3: 에이전트 루프 17종 상태 전이 및 자가 수정(Self-Correction Gate)

### 📌 에이전트 루프 상태 전이(State Machine)란?
Claude Code의 `queryLoop()`는 단순히 LLM의 답변을 기다리는 단순 반복문이 아닙니다. API 오버플로우, 터미널 인터럽트, 문법 에러, 토큰 예산 초과 등 **다양한 런타임 이벤트 상황에서 에이전트를 스스로 살려내거나(Continue) 안전하게 탈출(Exit)시키는 17가지 상태 전이(State Transition) 엔진**으로 동작합니다.

---

<div style="background-color: #0F172A; border: 1px solid #334155; border-radius: 12px; padding: 20px; font-family: 'Malgun Gothic', 'Pretendard', sans-serif; color: #E2E8F0; margin: 15px 0;">
<h3 style="color: #FBBF24; margin-top: 0; border-bottom: 2px solid #1E293B; padding-bottom: 10px; font-size: 1.15rem;">🛡️ Stop Hooks 자가 수정 게이트 (Self-Correction Gate)</h3>
<table style="width: 100%; border-collapse: separate; border-spacing: 5px; background: transparent; margin: 15px 0 0 0; border: none;">
<tr>
<td style="width: 30%; background-color: #1E293B; border: 1px solid #A78BFA; padding: 12px; border-radius: 8px; text-align: center; vertical-align: middle;">
<div style="font-weight: bold; color: #A78BFA; font-size: 0.9rem;">1. 에이전트 응답</div>
<div style="font-size: 0.8rem; color: #94A3B8; margin-top: 4px;">파이썬 코드 블록 생성</div>
</td>
<td style="width: 5%; color: #64748B; font-weight: bold; text-align: center; font-size: 1.2rem;">➔</td>
<td style="width: 30%; background-color: #1E293B; border: 1px solid #FBBF24; padding: 12px; border-radius: 8px; text-align: center; vertical-align: middle;">
<div style="font-weight: bold; color: #FBBF24; font-size: 0.9rem;">2. handleStopHooks()</div>
<div style="font-size: 0.8rem; color: #94A3B8; margin-top: 4px;">컴파일러 / 문법 검증</div>
</td>
<td style="width: 5%; color: #64748B; font-weight: bold; text-align: center; font-size: 1.2rem;">➔</td>
<td style="width: 30%; background-color: #064E3B; border: 1px solid #34D399; padding: 12px; border-radius: 8px; text-align: center; vertical-align: middle;">
<div style="font-weight: bold; color: #34D399; font-size: 0.9rem;">3. PASS (정상)</div>
<div style="font-size: 0.8rem; color: #A7F3D0; margin-top: 4px;">루프 안전 종료</div>
</td>
</tr>
</table>
<div style="background-color: #7F1D1D; border: 1px solid #EF4444; padding: 12px; border-radius: 8px; margin-top: 15px; text-align: center; font-size: 0.85rem; color: #FCA5A5;">
<b>🔴 FAIL (문법 오류 검출 시)</b>: 컴파일 에러 메시지(blockingError)를 대화에 자동 주입하고 queryLoop() 재진입 (자가 수정 재시도)
</div>
</div>

---


### 💡 오늘 실습에서 직접 만들어볼 핵심 메커니즘

이번 실습에서는 에이전트 루프(`queryLoop`)에서 예기치 못한 에러나 인터럽트 상황이 발생했을 때, 하네스가 어떻게 상태 전이(State Transition)를 일으켜 **루프를 자율 복구하거나 안전하게 정리(Graceful Shutdown)**하는지 4가지 핵심 핸들러로 직접 확인합니다.

#### 💡 실습할 4대 상태 전이 핸들러
1. **`stop_hook_blocking` (자가 수정 환류)**:
   - 에이전트가 문법 에러(`SyntaxError`)가 포함된 파이썬 코드를 생성했을 때, `StopHooksMiddleware`가 턴을 종결짓지 못하게 차단하고 **`blockingError`를 주입하여 자가 수정(Self-Repair)을 강제**합니다.
2. **`model_error` (통신 예외 지수 백오프 재시도)**:
   - LLM API 통신 장애(네트워크 에러, 500 장애) 발생 시, 에이전트가 즉시 크래시를 내지 않고 **지수 백오프(Exponential Backoff)로 자동 재시도** 후 정중한 안내 메시지로 세션을 보정합니다.
3. **`aborted_streaming` (응답 스트리밍 중단)**:
   - 응답 생성 중 사용자가 `Ctrl+C`나 Abort 버튼을 눌렀을 때, 대화 상태를 안전하게 적재하고 중단 노티피케이션을 남깁니다.
4. **`aborted_tools` (도구 실행 인터럽트)**:
   - 장시간 실행 중인 툴(`bash_command` 등) 실행 도중 중단 신호(Signal)를 수신하면, 툴 실행을 안전하게 취소(`aborted_tools`)하고 프로젝트 파손을 방지합니다.

In [ ]:
# -----------------------------------------------------------------------------
# ★ 핵심 실습 3: 5대 에러 코렉션 및 상태 전이 핸들러 실습 코드
# -----------------------------------------------------------------------------
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from modules.claude_code.self_correction import (
    StopHooksMiddleware,
    ModelErrorHandlerMiddleware,
    ModelFallbackMiddleware,
    AbortStreamingMiddleware,
    AbortStreamingHandler,
    AbortToolsMiddleware,
    AbortToolsHandler
)

In [ ]:
# =============================================================================
# 1. [stop_hook_blocking] 문법 에러 발생 시 blockingError 주입 및 자가 수정 강제
# =============================================================================
print("==========================================================")
print("🔴 1. [stop_hook_blocking] 문법 에러 코드 주입 테스트")
print("==========================================================")

stop_hook = StopHooksMiddleware()

# 에이전트가 괄호가 안 닫힌 잘못된 파이썬 코드를 생성한 상황
broken_code_msg = AIMessage(
    content="요청하신 파이썬 함수를 작성했습니다:\n```python\ndef calculate_total(price, tax:\n    return price + tax\n```"
)

state_broken = {"messages": [HumanMessage(content="DB 계산 함수 만들어줘"), broken_code_msg]}
result_broken = stop_hook.after_agent(state_broken)

print(f"📌 [검증 결과] 전이 상태(Transition): '{result_broken.get('transition')}'")
print("----------------------------------------------------------")
print("💬 [주입된 blockingError 메시지 확인 (마지막 메시지 pretty_print)]:")
result_broken["messages"][-1].pretty_print()


# =============================================================================
# 2. [completed] 정상 문법 코드 생성 시 검증 통과 및 루프 완결
# =============================================================================
print("\n" + "==========================================================")
print("🟢 2. [completed] 정상 코드 생성 시 Stop Hook 검증 통과")
print("==========================================================")

valid_code_msg = AIMessage(
    content="요청하신 올바른 파이썬 함수입니다:\n```python\ndef calculate_total(price, tax):\n    return price + tax\n```"
)

state_valid = {"messages": [HumanMessage(content="DB 계산 함수 만들어줘"), valid_code_msg]}
result_valid = stop_hook.after_agent(state_valid)

print(f"📌 [검증 결과] 전이 상태(Transition): '{result_valid.get('transition')}' (루프 안전 완결!)")


# =============================================================================
# 3. [aborted_streaming] 스트리밍 중 사용자 중단 (Ctrl+C / Abort) 처리
# =============================================================================
print("\n" + "==========================================================")
print("⚡ 3. [aborted_streaming] 스트리밍 중 사용자 중단 처리")
print("==========================================================")

stream_handler = AbortStreamingHandler()
messages_stream = [HumanMessage(content="대용량 리포트 생성해줘")]
aborted_stream_result = stream_handler.handle_abort(messages_stream)

print("💬 [안전하게 적재된 중단 메시지 pretty_print]:")
aborted_stream_result[-1].pretty_print()


# =============================================================================
# 4. [aborted_tools] 장시간 실행 툴 중 사용자 인터럽트 취소 처리
# =============================================================================
print("\n" + "==========================================================")
print("⚡ 4. [aborted_tools] bash_command 툴 실행 중 사용자 중단 취소")
print("==========================================================")

tool_handler = AbortToolsHandler()
messages_tool = [HumanMessage(content="전체 빌드 스크립트 실행해줘")]
aborted_tool_result = tool_handler.handle_tool_abort(messages_tool, active_tool_name="bash_command")

print("💬 [적재된 툴 취소 메시지 pretty_print]:")
aborted_tool_result[-1].pretty_print()


In [ ]:
# -----------------------------------------------------------------------------
# ★ 핵심 실습 3: 5대 에러 코렉션 및 상태 전이 핸들러 실습 코드
# -----------------------------------------------------------------------------
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from modules.claude_code.self_correction import (
    StopHooksMiddleware,
    ModelErrorHandlerMiddleware,
    ModelFallbackMiddleware,
    AbortStreamingMiddleware,
    AbortStreamingHandler,
    AbortToolsMiddleware,
    AbortToolsHandler
)

# =============================================================================
# 1. [stop_hook_blocking] 문법 에러 발생 시 blockingError 주입 및 자가 수정 강제
# =============================================================================
print("==========================================================")
print("🔴 1. [stop_hook_blocking] 문법 에러 코드 주입 테스트")
print("==========================================================")

stop_hook = StopHooksMiddleware()

# 에이전트가 괄호가 안 닫힌 잘못된 파이썬 코드를 생성한 상황
broken_code_msg = AIMessage(
    content="요청하신 파이썬 함수를 작성했습니다:\n```python\ndef calculate_total(price, tax:\n    return price + tax\n```"
)

state_broken = {"messages": [HumanMessage(content="DB 계산 함수 만들어줘"), broken_code_msg]}
result_broken = stop_hook.after_agent(state_broken)

print(f"📌 [검증 결과] 전이 상태(Transition): '{result_broken.get('transition')}'")
print("----------------------------------------------------------")
print("💬 [주입된 blockingError 메시지 확인 (마지막 메시지 pretty_print)]:")
result_broken["messages"][-1].pretty_print()


# =============================================================================
# 2. [completed] 정상 문법 코드 생성 시 검증 통과 및 루프 완결
# =============================================================================
print("\n" + "==========================================================")
print("🟢 2. [completed] 정상 코드 생성 시 Stop Hook 검증 통과")
print("==========================================================")

valid_code_msg = AIMessage(
    content="요청하신 올바른 파이썬 함수입니다:\n```python\ndef calculate_total(price, tax):\n    return price + tax\n```"
)

state_valid = {"messages": [HumanMessage(content="DB 계산 함수 만들어줘"), valid_code_msg]}
result_valid = stop_hook.after_agent(state_valid)

print(f"📌 [검증 결과] 전이 상태(Transition): '{result_valid.get('transition')}' (루프 안전 완결!)")


# =============================================================================
# 3. [aborted_streaming] 스트리밍 중 사용자 중단 (Ctrl+C / Abort) 처리
# =============================================================================
print("\n" + "==========================================================")
print("⚡ 3. [aborted_streaming] 스트리밍 중 사용자 중단 처리")
print("==========================================================")

stream_handler = AbortStreamingHandler()
messages_stream = [HumanMessage(content="대용량 리포트 생성해줘")]
aborted_stream_result = stream_handler.handle_abort(messages_stream)

print("💬 [안전하게 적재된 중단 메시지 pretty_print]:")
aborted_stream_result[-1].pretty_print()


# =============================================================================
# 4. [aborted_tools] 장시간 실행 툴 중 사용자 인터럽트 취소 처리
# =============================================================================
print("\n" + "==========================================================")
print("⚡ 4. [aborted_tools] bash_command 툴 실행 중 사용자 중단 취소")
print("==========================================================")

tool_handler = AbortToolsHandler()
messages_tool = [HumanMessage(content="전체 빌드 스크립트 실행해줘")]
aborted_tool_result = tool_handler.handle_tool_abort(messages_tool, active_tool_name="bash_command")

print("💬 [적재된 툴 취소 메시지 pretty_print]:")
aborted_tool_result[-1].pretty_print()


In [ ]:
from app.tools.common import file_read, file_writer, bash_command
from modules.claude_code.self_correction import (
    StopHooksMiddleware,
    ModelErrorHandlerMiddleware,
    ModelFallbackMiddleware,
    AbortStreamingHandler,
    AbortToolsHandler
)
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from app.utils.message_utils import normalize_content
from langchain.agents import create_agent

# 1. 툴 및 미들웨어 바인딩 초기화
tools = [file_read, file_writer, bash_command]

# 2. 5대 미들웨어 리스트 생성 및 create_agent 바인딩
agent = create_agent(
    model=llm,
    tools=tools,
    middleware=[
        StopHooksMiddleware(),
        ModelErrorHandlerMiddleware(max_retries=2, initial_delay=0.1),
        ModelFallbackMiddleware(fallback_model_name="gemini-2.5-pro"),
        AbortStreamingHandler(),
        AbortToolsHandler()
    ]
)
print("✅ Agent successfully initialized with all 5 self-correction middlewares!")


### 🧪 5대 Self-Correction 미들웨어 바인딩 에이전트 통합 테스트 (Agent Verification)

생성된 `agent` 객체에 대해 **1) 정상 질의 및 StopHooks 자동 검증**, **2) ModelFallback 메인 모델 장애 스위칭**, **3) StopHooks 문법 에러 감지 및 자가 수정(Self-Repair) 환류** 3가지 테스트 시나리오를 직접 실행하여 미들웨어 동작을 검증합니다.

In [ ]:
# -----------------------------------------------------------------------------
# 🧪 [테스트 1] 기본 질의 실행 및 미들웨어 통합 검증 (StopHooks 자동 통과)
# -----------------------------------------------------------------------------
print("==========================================================")
print("🤖 1. 에이전트 정상 질의 수행 & 미들웨어 파이프라인 검증")
print("==========================================================")

query = "두 숫자의 합을 구하는 calculate_sum(a, b) 파이썬 함수를 작성하고 짧게 설명해줘."
result = agent.invoke({"messages": [HumanMessage(content=query)]})

print("\n💬 [에이전트 최종 응답 메시지 pretty_print]:")
result["messages"][-1].pretty_print()


In [ ]:
# -----------------------------------------------------------------------------
# 🧪 [테스트 2] ModelFallbackMiddleware 메인 모델 장애 발생 시 백업 모델 스위칭 검증
# -----------------------------------------------------------------------------
print("==========================================================")
print("🔄 2. ModelFallbackMiddleware 장애 발생 시 대체 모델 스위칭 검증")
print("==========================================================")

# 1. 의도적으로 primary LLM 장애(ConnectionError/500)를 유발하는 Mock 모델 정의
class BrokenPrimaryModel:
    def invoke(self, *args, **kwargs):
        raise ConnectionError("Primary LLM (gemini-3.5-flash) 500 Connection Timeout Error")
    def bind_tools(self, *args, **kwargs):
        return self

# 2. 장애 시 대체(Fallback)로 작동할 백업 모델 정의
class BackupFallbackModel:
    def invoke(self, *args, **kwargs):
        return AIMessage(content="[Fallback 성공] 안녕하세요! 대체 모델(gemini-2.5-pro)로 정상 전환되어 계산 결과를 응답합니다.")
    def bind_tools(self, *args, **kwargs):
        return self

broken_llm = BrokenPrimaryModel()
mock_backup_llm = BackupFallbackModel()

# 3. Primary LLM 장애 상황으로 에이전트 바인딩
fallback_agent = create_agent(
    model=broken_llm,
    tools=tools,
    middleware=[
        ModelFallbackMiddleware(fallback_model_name="gemini-2.5-pro", fallback_llm=mock_backup_llm),
        StopHooksMiddleware(),
    ]
)

print("⚡ Primary Model 장애 유발 질의를 전송합니다...")
res_fallback = fallback_agent.invoke({
    "messages": [HumanMessage(content="안녕하세요! 1+1 계산 결과를 답해주세요.")]
})

print("\n💬 [ModelFallback 스위칭 후 최종 응답 pretty_print]:")
res_fallback["messages"][-1].pretty_print()


In [ ]:
# -----------------------------------------------------------------------------
# 🧪 [테스트 3] StopHooksMiddleware 문법 에러 감지 & 자가 수정(Self-Repair) 환류 검증
# -----------------------------------------------------------------------------
print("==========================================================")
print("🔴 3. StopHooksMiddleware 문법 에러 감지 & 자가 수정(Self-Repair) 환류 검증")
print("==========================================================")

broken_code_msg = AIMessage(
    content="작성한 코드입니다:\n```python\ndef bad_func(x, y:\n    return x + y\n```"
)
state_syntax_test = {
    "messages": [
        HumanMessage(content="bad_func 만들어줘"),
        broken_code_msg
    ]
}

stop_hook_mw = StopHooksMiddleware()
corrected_state = stop_hook_mw.after_agent(state_syntax_test)

print(f"📌 [검증 결과] 전이 상태: '{corrected_state.get('transition')}'")
print("\n💬 [주입된 blockingError 메시지 pretty_print]:")
corrected_state["messages"][-1].pretty_print()


In [ ]:
# -----------------------------------------------------------------------------
# 🧪 [테스트 4] AbortStreamingMiddleware 사용자의 응답 스트리밍 중단(Ctrl+C) 감지 테스트
# -----------------------------------------------------------------------------
print("==========================================================")
print("⚡ 4. AbortStreamingMiddleware 응답 스트리밍 사용자 중단(Ctrl+C) 감지 테스트")
print("==========================================================")

# 사용자가 응답 스트리밍 도중 Ctrl+C / Abort 버튼을 누른 상황 시뮬레이션
class AbortedModel:
    def invoke(self, *args, **kwargs):
        raise KeyboardInterrupt("User pressed Ctrl+C during streaming")
    def bind_tools(self, *args, **kwargs):
        return self

aborted_agent = create_agent(
    model=AbortedModel(),
    tools=tools,
    middleware=[
        AbortStreamingHandler()
    ]
)

print("⚡ 모델 응답 생성 중 Ctrl+C(KeyboardInterrupt) 중단 신호 발생...")
res_aborted = aborted_agent.invoke({
    "messages": [HumanMessage(content="대용량 보고서를 생성해줘.")]
})

print("\n💬 [AbortStreamingMiddleware 가로채기 성공 후 안전 메시지 pretty_print]:")
res_aborted["messages"][-1].pretty_print()


In [ ]:
# -----------------------------------------------------------------------------
# 🧪 [테스트 5] AbortToolsMiddleware 장시간 실행 툴(bash_command) 사용자 중단 감지 테스트
# -----------------------------------------------------------------------------
print("==========================================================")
print("⚡ 5. AbortToolsMiddleware 장시간 실행 툴(bash_command) 사용자 중단 감지 테스트")
print("==========================================================")

from langchain_core.tools import tool

# 장시간 실행 도중 사용자가 중단 신호를 보낸 툴 시뮬레이션
@tool
def long_running_bash(cmd: str) -> str:
    """Long running bash command."""
    raise KeyboardInterrupt("User interrupted long-running tool execution")

class ToolCallingModel:
    def __init__(self):
        self.call_count = 0
    def invoke(self, messages, *args, **kwargs):
        self.call_count += 1
        if self.call_count == 1:
            return AIMessage(
                content="Long running bash command...",
                tool_calls=[{"name": "long_running_bash", "args": {"cmd": "sleep 100"}, "id": "call_bash_001"}]
            )
        return AIMessage(content="툴 실행 취소 확인 후 다음 단계를 진행합니다.")
    def bind_tools(self, *args, **kwargs):
        return self

aborted_tool_agent = create_agent(
    model=ToolCallingModel(),
    tools=[long_running_bash],
    middleware=[
        AbortToolsHandler()
    ]
)

print("⚡ 툴 실행 중 인터럽트 중단 신호 발생...")
res_tool_aborted = aborted_tool_agent.invoke({
    "messages": [HumanMessage(content="빌드 스크립트 실행해줘")]
})

print("\n💬 [AbortToolsMiddleware 툴 중단 메시지 적재 확인 pretty_print]:")
res_tool_aborted["messages"][2].pretty_print()
